In [1]:
from bs4 import XMLParsedAsHTMLWarning
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

import sys
sys.path.append("..")  # src 모듈을 import 하기 위해 상위 폴더를 경로에 추가

from src.edgar_client import fetch_filing_html
from src.parser import find_schedule_of_investments_tables, _row_cells, parse_filing_html

url = "https://www.sec.gov/Archives/edgar/data/1736035/000173603526000010/bxsl-20260331.htm"
html = fetch_filing_html(url)

tables = find_schedule_of_investments_tables(html)
print(f"찾은 테이블 개수: {len(tables)}")

찾은 테이블 개수: 86


In [2]:
from src.parser import find_schedule_of_investments_tables, _table_as_of_date
from src.edgar_client import extract_period_end_date

tables = find_schedule_of_investments_tables(html)
for i, table in enumerate(tables):
    print(i, _table_as_of_date(table))

0 2026-03-31
1 2026-03-31
2 2026-03-31
3 2026-03-31
4 2026-03-31
5 2026-03-31
6 2026-03-31
7 2026-03-31
8 2026-03-31
9 2026-03-31
10 2026-03-31
11 2026-03-31
12 2026-03-31
13 2026-03-31
14 2026-03-31
15 2026-03-31
16 2026-03-31
17 2026-03-31
18 2026-03-31
19 2026-03-31
20 2026-03-31
21 2026-03-31
22 2026-03-31
23 2026-03-31
24 2026-03-31
25 2026-03-31
26 2026-03-31
27 2026-03-31
28 2026-03-31
29 2026-03-31
30 2026-03-31
31 2026-03-31
32 2026-03-31
33 None
34 None
35 2026-03-31
36 2026-03-31
37 2026-03-31
38 2026-03-31
39 2026-03-31
40 2026-03-31
41 2026-03-31
42 None
43 2026-03-31
44 2025-12-31
45 2025-12-31
46 2025-12-31
47 2025-12-31
48 2025-12-31
49 2025-12-31
50 2025-12-31
51 2025-12-31
52 2025-12-31
53 2025-12-31
54 2025-12-31
55 2025-12-31
56 2025-12-31
57 2025-12-31
58 2025-12-31
59 2025-12-31
60 2025-12-31
61 2025-12-31
62 2025-12-31
63 2025-12-31
64 2025-12-31
65 2025-12-31
66 2025-12-31
67 2025-12-31
68 2025-12-31
69 2025-12-31
70 2025-12-31
71 2025-12-31
72 2025-12-31
73 202

In [ ]:
from src.pipeline import build_time_series, save_to_sqlite
from config import BXSL_CIK

parsed, unmatched = build_time_series(BXSL_CIK, form_types=["10-Q", "10-K"], limit=8)
print(parsed.shape, unmatched.shape)

save_to_sqlite(parsed, unmatched)

Fetching 2026-05-07 (10-Q), period end: 2026-03-31...
[parser] 8 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2026-02-25 (10-K), period end: 2025-12-31...
[parser] 8 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-11-10 (10-Q), period end: 2025-09-30...
[parser] 7 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-08-06 (10-Q), period end: 2025-06-30...
[parser] 10 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-05-07 (10-Q), period end: 2025-03-31...
[parser] 2 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-02-26 (10-K), period end: 2024-12-31...
[parser] 2 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2024-11-12 (10-Q), perio

In [ ]:
from src.edgar_client import get_recent_filings, build_document_url, fetch_filing_html
from src.parser import find_schedule_of_investments_tables, _table_as_of_date, _row_cells
from config import BXSL_CIK

# 2024-06-30 filing만 다시 열어서 Aevex 관련 원본 행 확인
filings = get_recent_filings(BXSL_CIK, form_types=["10-Q"], limit=8)
target = [f for f in filings if f["filing_date"] == "2024-08-07"][0]
url = build_document_url(target["cik"], target["accession_number"], target["primary_document"])
html_2024q2 = fetch_filing_html(url)

tables_2024q2 = find_schedule_of_investments_tables(html_2024q2)
current_2024q2 = [t for t in tables_2024q2 if _table_as_of_date(t) == "2024-06-30"]

for table in current_2024q2:
    for tr in table.find_all("tr"):
        cells = _row_cells(tr)
        if any("Aevex" in c for c in cells):
            print(cells)

NameError: name 'get_recent_filings' is not defined

In [4]:
parsed[parsed["investment_name"] == "Aevex Holdings, LLC"][
    ["investment_name", "fair_value", "period_end_date", "acquisition_date"]
].sort_values("period_end_date")

,investment_name,fair_value,period_end_date,acquisition_date
6364,"Aevex Holdings, LLC","47,548",2024-06-30,11.44%
5632,"Aevex Holdings, LLC","47,428",2024-09-30,4/30/2024
4794,"Aevex Holdings, LLC","47,309",2024-12-31,4/30/2024
3938,"Aevex Holdings, LLC","46,967",2025-03-31,4/30/2024
3027,"Aevex Holdings, LLC","46,835",2025-06-30,4/30/2024
2043,"Aevex Holdings, LLC","46,717",2025-09-30,4/30/2024
2044,"Aevex Holdings, LLC","107,690",2025-09-30,3/17/2020
1020,"Aevex Holdings, LLC","46,598",2025-12-31,4/30/2024
1021,"Aevex Holdings, LLC","107,406",2025-12-31,3/17/2020
0,"Aevex Holdings, LLC","46,713",2026-03-31,4/30/2024
